<div style="display: flex; align-items: center; gap: 8px;">
  <img src="images/microsoft-symbol.svg" alt="Microsoft" style="width: 80px; height: 80px; border-radius: 8px; flex: 0 0 auto;">
  <h1 style="color: #4A2D6F; font-weight: 800; margin: 0;">Microsoft Foundry: Agent Framework + OTEL — PoC</h1>
</div>
<p align="center" style="color: #666; font-style: italic; margin-top: -4px;">End-to-end proof of concept — from agent execution to trace validation in Log Analytics</p>

Build and interact with an AI agent using Microsoft Agent Framework, Microsoft Foundry, and OpenTelemetry. </br>
The notebook is intentionally split into two execution paths:

1. **Storytelling + Microsoft Learn via MCP** — routed through the latest Microsoft Agent Framework SDKs.
2. **Microsoft Sentinel Data Exploration via MCP** — preserved on the existing Foundry project-connected MCP dependency by design.

For observability, the notebook wires up **Azure Monitor / OpenTelemetry tracing** to send agentic telemetry to:

- Microsoft Foundry — Tracing (Preview)
- Application Insights
- Interrogate Log Analytics (AppDependencies)

---

<h2 style="color: #D83B01;">Prerequisites</h2>

<details>
<summary><strong>Expand Prerequisites</strong></summary>

Before running this notebook, ensure the following are in place:

**1. Azure CLI Installed**

The notebook uses `DefaultAzureCredential`, which relies on Azure CLI for local authentication.

- **Install via winget (Windows):**
  ```powershell
  winget install --id Microsoft.AzureCLI -e --accept-source-agreements --accept-package-agreements
  ```
- **Other platforms / manual install:** [https://aka.ms/installazurecli](https://aka.ms/installazurecli)

**2. Logged in via Azure CLI with Appropriate Permissions**

After installing, sign in and verify your session:
```bash
az login
az account show
```
Your Entra ID identity must have **Contributor** (or equivalent) access to the Microsoft Foundry project and the associated Application Insights resource.

**3. Microsoft Foundry Project with Application Insights**

You need a project provisioned in **Microsoft Foundry** that is connected to an **Application Insights** instance backed by a **Log Analytics workspace**. The notebook retrieves the Application Insights connection string from the project at runtime to enable telemetry export.

**4. Microsoft Sentinel MCP Project Connection**

The Microsoft Sentinel section intentionally depends on an existing Foundry **project connection** for the Sentinel MCP endpoint. Keep that connection in place if you want the Sentinel cells to run.

</details>

---


<h2 style="color: #0078D4;">0. Create or Reuse Virtual Environment &amp; Register Kernel</h2>

<ul>
  <li>Create (or reuse) a Python virtual environment to isolate dependencies for this project, then install <code>ipykernel</code> and register it as a notebook kernel.</li>
  <li>Once setup is complete, select <strong>Kernel</strong> and choose your new <code>.venv</code> environment.</li>
</ul>


In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), ".venv")

# Create the virtual environment (safe to run repeatedly)
subprocess.check_call([sys.executable, "-m", "venv", venv_dir])

# Resolve the venv Python path per OS, then use "python -m pip" for reliability
venv_python = (
    os.path.join(venv_dir, "Scripts", "python.exe")
    if os.name == "nt"
    else os.path.join(venv_dir, "bin", "python")
)

# Install / upgrade ipykernel inside the venv
subprocess.check_call([venv_python, "-m", "pip", "install", "--upgrade", "ipykernel"])

# Register the venv as a Jupyter kernel
subprocess.check_call([
    venv_python,
    "-m",
    "ipykernel",
    "install",
    "--user",
    "--name",
    "ai-agent-demo",
    "--display-name",
    "AI Agent Demo (.venv)",
])

print(f"Virtual environment created or reused at {venv_dir}")
print("Select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker, then continue.")

<h3 style="color: #D83B01;">Activate or Deactivate the Python Virtual Environment</h3>

Run the cell below to **activate** the <code style="color: #2EA043;">.venv</code> so it is recognized as an available kernel.

You only need to create the environment once (Section 0), but you may need to re-activate it after restarting VS Code or opening a new terminal session.

To **deactivate**, run <code style="color: #D29922;">deactivate</code> in your PowerShell terminal.


In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), ".venv")
venv_dir_norm = os.path.normcase(os.path.normpath(venv_dir))

active_venv = os.environ.get("VIRTUAL_ENV", "")
active_venv_norm = os.path.normcase(os.path.normpath(active_venv)) if active_venv else ""

already_active = (
    active_venv_norm == venv_dir_norm
    or os.path.normcase(os.path.normpath(sys.prefix)) == venv_dir_norm
)

if already_active:
    print(f"ℹ️ .venv is already active: {venv_dir}")
else:
    # Resolve the activation script path per OS
    if os.name == "nt":
        activate_script = os.path.join(venv_dir, "Scripts", "Activate.ps1")
        subprocess.check_call(["powershell", "-ExecutionPolicy", "Bypass", "-File", activate_script])
    else:
        activate_script = os.path.join(venv_dir, "bin", "activate")
        subprocess.check_call(["bash", "-c", f"source {activate_script}"])

    print(f"✅ Virtual environment activated: {activate_script}")

print("You can now select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker.")

<h2 style="color: #0078D4;">1. Install Python Packages &amp; Dependencies</h2>

Install the required Python packages.

<details>
<summary><strong>Package Overview (Expand to view dependencies)</strong></summary>
<br>

🤖 <code style="color: #0078D4; font-weight: 600;">agent-framework-core</code>
 — Core Microsoft Agent Framework package for shared helpers used by the notebook.<br>
💬 <code style="color: #0078D4; font-weight: 600;">agent-framework-openai</code>
 — Current Agent Framework OpenAI package retained for shared observability helpers.<br>
📦 <code style="color: #0078D4; font-weight: 600;">azure-ai-projects</code>
 — Used for Foundry project metadata, agent creation, the Application Insights connection string, and the Sentinel MCP dependency.<br>
🔐 <code style="color: #0078D4; font-weight: 600;">azure-identity</code>
 — Provides <code>DefaultAzureCredential</code> for seamless Azure authentication via Entra ID.<br>
📡 <code style="color: #0E7C6B; font-weight: 600;">opentelemetry-sdk</code>
 — Core OpenTelemetry SDK for instrumentation and tracing.<br>
📊 <code style="color: #0078D4; font-weight: 600;">azure-monitor-opentelemetry</code>
 — Azure Monitor exporter and tracing setup for notebook telemetry.<br>
🔗 <code style="color: #0E7C6B; font-weight: 600;">opentelemetry-instrumentation-httpx</code>
 — Adds HTTPX client instrumentation, which the notebook pairs with explicit `responses.create(...)` client spans so Service Map can draw a client-to-service edge.<br>
&emsp;↳ <code style="color: #888; font-weight: 500;">azure-core-tracing-opentelemetry</code>
 <span style="color: #666; font-size: 0.9em;">Azure SDK tracing bridge.</span>

</details>

In [ ]:
import json
import subprocess
import sys

outdated = subprocess.check_output(
    [sys.executable, "-m", "pip", "list", "--outdated", "--format=json"],
    text=True,
)
if any(pkg["name"].lower() == "pip" for pkg in json.loads(outdated)):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

%pip --disable-pip-version-check install --upgrade --pre azure-monitor-opentelemetry-exporter
%pip --disable-pip-version-check install --upgrade "agent-framework-core>=1.0.0" "agent-framework-openai>=1.0.0" "azure-ai-projects>=2.0.0b4"
%pip --disable-pip-version-check install --upgrade azure-identity azure-monitor-opentelemetry azure-core-tracing-opentelemetry opentelemetry-sdk opentelemetry-instrumentation-httpx

<h3 style="color: #D83B01;">Confirm Existing Azure / Microsoft Foundry Deployment</h3>

Load the latest <code style="color: #2EA043;">build_info-&lt;suffix&gt;.json</code> and print the current infrastructure summary before importing the Azure/Foundry libraries.

<table style="margin-top: 8px; border: none; border-collapse: collapse;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📂</td>
    <td style="border: none; padding: 4px 0; color: #555;">Reads deployment metadata (resource group, Key Vault, App Insights, model, endpoints)</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">✅</td>
    <td style="border: none; padding: 4px 0; color: #555;">Confirms all resources are provisioned before proceeding</td>
  </tr>
</table>


In [ ]:
import json
from pathlib import Path

def resolve_build_info_path() -> Path:
    candidates = sorted(
        Path(".").glob("build_info-*.json"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        r"No build_info-<suffix>.json file found. Run deployment/deploy-foundry-env.ps1 first to generate one."
    )

build_info_path = resolve_build_info_path()
build_info = json.loads(build_info_path.read_text(encoding="utf-8"))
foundry_proj_ep = build_info["foundry_project_endpoint"]
genai_model = build_info["genai_model"]
signed_in_account = globals().get("signed_in_account")

def show_current_build_status(
    build_info: dict,
    *,
    signed_in_account: str | None = None,
    app_insights_connection_status: str = "This project only ✅",
    law_rbac_status: str = "Log Analytics Reader on DIBSecCom ✅",
) -> None:
    user_status = (
        f"{signed_in_account} added to zolab-ai-dev ✅"
        if signed_in_account
        else "Signed-in account not available"
    )
    rows = [
        ("☁️ Resource Group", f"✅ {build_info['rg']}"),
        ("🗄️ Storage", build_info["storage_account"]),
        ("🔐 Key Vault", build_info["key_vault"]),
        ("📊 App Insights", build_info["appinsights"]),
        ("🤖 AI Foundry", build_info["foundry_name"]),
        ("🏢 AI Project", build_info["foundry_project_name"]),
        ("🧠 Model", f"{build_info['genai_model']} (GlobalStandard)"),
        ("📝 Build Info", build_info_path.name),
        ("🔗 App Insights Connection", app_insights_connection_status),
        ("📡 LAW RBAC", law_rbac_status),
        ("👤 User", user_status),
        ("🔌 Foundry Project Endpoint", build_info["foundry_project_endpoint"]),
        ("🤖 Azure OpenAI Endpoint", build_info["azure_openai_endpoint"]),
    ]

    item_width = max(29, max(len(item) for item, _ in rows) + 2)
    status_width = max(78, max(len(status) for _, status in rows) + 2)

    print()
    print("● ☁️🎉🚀 Your infrastructure is all green!")
    print()
    print("┌" + ("─" * item_width) + "┬" + ("─" * status_width) + "┐")
    print("│" + " Item".ljust(item_width) + "│" + " Status".ljust(status_width) + "│")
    print("├" + ("─" * item_width) + "┼" + ("─" * status_width) + "┤")
    for item, status in rows:
        print("│" + f" {item}".ljust(item_width) + "│" + f" {status}".ljust(status_width) + "│")
    print("└" + ("─" * item_width) + "┴" + ("─" * status_width) + "┘")
    print()
    print("Ready for the notebook! 🎯")

show_current_build_status(build_info, signed_in_account=signed_in_account)


<h2 style="color: #0078D4;">2. Import Libraries</h2>

Import the necessary classes from the Azure, Foundry, OpenTelemetry, and supporting SDKs.

<details>
<summary><strong>What Each Import Does</strong></summary>
<br>

🔐 <code style="color: #0078D4; font-weight: 600;">DefaultAzureCredential</code>
 <span style="color: #888; font-size: 0.88em;">azure-identity</span>
 — Provides local developer authentication via Azure CLI and Azure-hosted authentication via managed identity.<br>
🤖 <code style="color: #0078D4; font-weight: 600;">AIProjectClient</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Client for Foundry project metadata, agent definitions, telemetry connection details, and the retained Sentinel MCP dependency.<br>
✨ <code style="color: #0078D4; font-weight: 600;">PromptAgentDefinition</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Declarative agent definition used for both the main storytelling/Learn agent and the Sentinel agent.<br>
🔧 <code style="color: #0078D4; font-weight: 600;">MCPTool</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Attaches either the public Microsoft Learn MCP endpoint or the retained Sentinel project connection to a Foundry project agent.<br>
📡 <code style="color: #0E7C6B; font-weight: 600;">AIProjectInstrumentor</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Enables preview GenAI tracing for project-backed agent creation and <code>responses.create(...)</code> calls.<br>
📊 <code style="color: #0E7C6B; font-weight: 600;">configure_azure_monitor</code>
 <span style="color: #888; font-size: 0.88em;">azure-monitor-opentelemetry</span>
 — Sets up Azure Monitor as the OpenTelemetry export backend.<br>
🧭 <code style="color: #666; font-weight: 600;">create_resource</code>
 <span style="color: #888; font-size: 0.88em;">agent-framework-core</span>
 — Provides a convenient OpenTelemetry resource helper for the notebook's Azure Monitor exporter setup.

</details>

<details>
<summary><strong>How the Pieces Wire Together</strong></summary>
<br>

The notebook now connects Foundry, the Responses API, and the telemetry stack in four steps:<br>

<span style="color: #0078D4; font-weight: 700;">①</span>
 <code style="color: #0078D4;">project_client.telemetry.get_application_insights_connection_string()</code>
 — retrieves the telemetry destination from the Foundry project.<br>
<span style="color: #0078D4; font-weight: 700;">②</span>
 <code style="color: #0078D4;">configure_azure_monitor(...)</code>
 — initializes the Azure Monitor OpenTelemetry pipeline.<br>
<span style="color: #0078D4; font-weight: 700;">③</span>
 <code style="color: #0E7C6B;">AIProjectInstrumentor().instrument()</code>
 — enables preview GenAI tracing for project-backed agent operations.<br>
<span style="color: #0078D4; font-weight: 700;">④</span>
 <code style="color: #D83B01;">responses.create(..., extra_body={"agent_reference": ...})</code>
 — runs the main and Sentinel agents through the Foundry Responses API so they surface in Foundry Agent Traces.<br>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🏗️</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Foundry SDK</strong> — project setup, telemetry connection string retrieval, agent version creation, and MCP tool attachment.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🛰️</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Responses API</strong> — executes both the main storytelling/Microsoft Learn path and the Sentinel path as project-backed agent runs.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Telemetry Stack</strong> — <code>opentelemetry-sdk</code> (tracing runtime) → <code>azure-core-tracing-opentelemetry</code> (bridge) → <code>azure-monitor-opentelemetry</code> (exporter).</td>
  </tr>
</table>

</details>

<details>
<summary><strong>MSFT Learn References</strong></summary>

- [Foundry client-side tracing (preview)](https://learn.microsoft.com/azure/foundry/observability/how-to/trace-agent-client-side)
- [Azure Monitor](https://learn.microsoft.com/python/api/overview/azure/monitor-opentelemetry-readme?view=azure-python#getting-started)
- [Azure AI Projects SDK](https://learn.microsoft.com/python/api/azure-ai-projects/azure.ai.projects.aiprojectclient?view=azure-python)

</details>


In [ ]:
import sys, platform

from agent_framework.observability import create_resource
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.ai.projects.telemetry import AIProjectInstrumentor
from azure.identity import DefaultAzureCredential

print("✅ All libraries imported successfully")
print("   📦 Runtime:  Azure AI Projects + Responses API")
print(f"   💻 Platform: {platform.system()} {platform.release()}")
print(f"   🐍 Python:   {sys.version.split()[0]}")

<h2 style="color: #0078D4;">3. Configure Credentials and Project Client</h2>

- Reuse the deployment values loaded in the **Confirm Existing Deployment** section above.
- Create an authenticated `AIProjectClient` for project metadata, main agent execution, and the retained Sentinel MCP dependency.
- Prefer local developer credentials when the notebook is not running on Azure so managed identity probing does not call the IMDS endpoint.
- Confirm which credential was selected for this notebook session.


In [ ]:
import logging
import os
import shutil
import subprocess

# build_info and build_info_path are set in Section 0
foundry_proj_ep = build_info["foundry_project_endpoint"]
azure_openai_endpoint = build_info["azure_openai_endpoint"]
genai_model = build_info["genai_model"]
signed_in_account = globals().get("signed_in_account")

azure_host_markers = (
    "WEBSITE_SITE_NAME",
    "WEBSITE_HOSTNAME",
    "FUNCTIONS_WORKER_RUNTIME",
    "KUBERNETES_SERVICE_HOST",
    "AKS_ARM_NAMESPACE_ID",
)
running_on_azure_host = any(os.environ.get(name) for name in azure_host_markers)

def _build_default_azure_credential(*, prefer_local_credentials: bool = False):
    credential_kwargs = {"process_timeout": 30}
    managed_identity_client_id = os.environ.get("AZURE_CLIENT_ID", "").strip()

    if running_on_azure_host and managed_identity_client_id:
        credential_kwargs["managed_identity_client_id"] = managed_identity_client_id

    if prefer_local_credentials and not running_on_azure_host:
        credential_kwargs["exclude_managed_identity_credential"] = True
        credential_kwargs["exclude_workload_identity_credential"] = True

    return DefaultAzureCredential(**credential_kwargs)

# ---------------------------------------------------------------------------
def _install_az_cli() -> bool:
    """Attempt to install Azure CLI using winget. Returns True on success."""
    if os.name != "nt":
        print("❌ Automatic install via winget is only supported on Windows.")
        print("   👉 Install Azure CLI from: https://aka.ms/installazurecli")
        return False

    if not shutil.which("winget"):
        print("❌ winget is not available on this system.")
        print("   👉 Install Azure CLI manually from: https://aka.ms/installazurecli")
        return False

    print("📦 Installing Azure CLI via winget … (this may take a few minutes)")
    result = subprocess.run(
        ["winget", "install", "--id", "Microsoft.AzureCLI", "-e", "--accept-source-agreements", "--accept-package-agreements"],
        capture_output=True, text=True, timeout=300,
    )
    if result.returncode == 0:
        print("✅ Azure CLI installed successfully.")
        az_default = os.path.join(os.environ.get("ProgramFiles", r"C:\Program Files"), "Microsoft SDKs", "Azure", "CLI2", "wbin")
        if os.path.isdir(az_default) and az_default not in os.environ.get("PATH", ""):
            os.environ["PATH"] = az_default + os.pathsep + os.environ["PATH"]
            print(f"   Added {az_default} to PATH for this session.")
        return True

    print(f"❌ winget install failed (exit code {result.returncode}).")
    if result.stderr:
        print(f"   {result.stderr.strip()}")
    print("   👉 Try installing manually from: https://aka.ms/installazurecli")
    return False


# ---------------------------------------------------------------------------
# Helper: check for Azure CLI and attempt interactive login if needed
# ---------------------------------------------------------------------------
def _ensure_az_login() -> bool:
    """Return True if Azure CLI is available and the user is logged in."""
    az_exe = "az.cmd" if os.name == "nt" else "az"

    if not shutil.which(az_exe):
        print("⚠️  Azure CLI is NOT installed.")
        if not _install_az_cli():
            return False
        if not shutil.which(az_exe):
            print("❌ Azure CLI still not found on PATH after install.")
            print("   Please restart VS Code so the updated PATH takes effect, then re-run this cell.")
            return False

    print("🔍 Azure CLI found on PATH — checking login status …")

    check = subprocess.run(
        [az_exe, "account", "show"],
        capture_output=True, text=True, timeout=15,
    )
    if check.returncode == 0:
        print("✅ Already logged in to Azure CLI.")
        return True

    print("⚠️  No active Azure CLI session detected. Launching 'az login' …")
    print("   A browser window should open — please sign in with your Azure account.\n")
    login = subprocess.run([az_exe, "login"], capture_output=True, text=True, timeout=120)
    if login.returncode == 0:
        if login.stdout:
            print(login.stdout.strip())
        print("\n✅ Azure CLI login succeeded.")
        return True

    print(f"\n❌ 'az login' did not complete successfully (exit code {login.returncode}).")
    if login.stderr:
        print(f"   {login.stderr.strip()}")
    print("   Please run 'az login' manually in a terminal, then re-run this cell.")
    return False


# ---------------------------------------------------------------------------
# Authenticate
# ---------------------------------------------------------------------------
credential = _build_default_azure_credential(prefer_local_credentials=True)

_identity_handler = logging.StreamHandler()
identity_logger = logging.getLogger("azure.identity")
identity_logger.addHandler(_identity_handler)
identity_logger.setLevel(logging.INFO)

signed_in_account = None

try:
    _ = credential.get_token("https://management.azure.com/.default")

except Exception as auth_ex:
    print(f"⚠️  DefaultAzureCredential failed: {type(auth_ex).__name__}")
    print("   Falling back to Azure CLI authentication …\n")

    if not _ensure_az_login():
        raise SystemExit(
            "🛑 Cannot proceed without valid Azure credentials. "
            "Please install/login to Azure CLI and re-run this cell."
        )

    credential = _build_default_azure_credential(prefer_local_credentials=True)
    _ = credential.get_token("https://management.azure.com/.default")

finally:
    identity_logger.removeHandler(_identity_handler)
    identity_logger.setLevel(logging.WARNING)

project_client = AIProjectClient(
    endpoint=foundry_proj_ep,
    credential=credential,
)

os.environ["FOUNDRY_PROJECT_ENDPOINT"] = foundry_proj_ep
os.environ["FOUNDRY_MODEL_NAME"] = genai_model
os.environ["AZURE_AI_PROJECT_ENDPOINT"] = foundry_proj_ep
os.environ["AZURE_OPENAI_ENDPOINT"] = azure_openai_endpoint
os.environ["AZURE_OPENAI_MODEL"] = genai_model
os.environ["AZURE_OPENAI_CHAT_MODEL"] = genai_model
os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"] = genai_model
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"

_lw = 24
print("\n✅ Notebook clients configured")
print(f"   🌐 {'Project endpoint:':<{_lw}} {foundry_proj_ep}")
print(f"   ☁️ {'Azure OpenAI:':<{_lw}} {azure_openai_endpoint}")
print(f"   🧠 {'Model deployment:':<{_lw}} {genai_model}")
print(f"   📄 {'Build info:':<{_lw}} {build_info_path.resolve()}")
print(f"   🔐 {'Auth:':<{_lw}} DefaultAzureCredential")
print(f"   🤖 {'Main runtime:':<{_lw}} Foundry project agent + Responses API")
print(f"   🔧 {'Learn tool path:':<{_lw}} Public MCPTool on main project agent")
print(f"   🛰️ {'Sentinel path:':<{_lw}} Foundry project agent + MCP project connection")
print(
    f"   🧭 {'Local credential mode:':<{_lw}} "
    + (
        "Managed identity excluded to avoid IMDS probes"
        if not running_on_azure_host
        else "Azure-host credential chain enabled"
    )
)


# ---------------------------------------------------------------------------
# Optional: surface which credential + identity is in use
# ---------------------------------------------------------------------------
def _run_command(command: list[str]) -> str | None:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if result.returncode == 0:
            value = (result.stdout or "").strip()
            return value or None
    except Exception:
        pass
    return None


def _resolve_identity_hint(credential_name: str) -> str | None:
    if credential_name == "AzureCliCredential":
        az_exe = "az.cmd" if os.name == "nt" else "az"
        return _run_command([az_exe, "account", "show", "--query", "user.name", "-o", "tsv"])
    if credential_name == "AzurePowerShellCredential":
        powershell_cmd = "$ctx = Get-AzContext; if ($ctx -and $ctx.Account) { $ctx.Account.Id }"
        return _run_command(["pwsh", "-NoProfile", "-Command", powershell_cmd])
    return None


try:
    selected_credential = getattr(credential, "_successful_credential", None)
    if selected_credential is not None:
        credential_name = selected_credential.__class__.__name__
        credential_name_color = globals().get("ansi_magenta", "\033[95m")
        credential_ansi_reset = globals().get("ansi_reset", "\033[0m")
        credential_name_colored = f"{credential_name_color}{credential_name}{credential_ansi_reset}"
        print(f"      🔐 Credential used: {credential_name_colored}")

        identity_hint = _resolve_identity_hint(credential_name)
        if identity_hint:
            signed_in_account = identity_hint
            print(f"      👤 Signed-in account: {identity_hint}")
        else:
            print("      👤 Signed-in account: not available for this credential type")
    else:
        print("      Credential Type: check azure.identity INFO logs above")
except Exception as ex:
    print(f"      Credential probe skipped: {type(ex).__name__}: {ex}")

<h2 style="color: #0078D4;">3.1 Enable Foundry Client-side Observability/Telemetry</h2>

<details>
<summary><strong>Tracing Setup Overview</strong></summary>

Configure tracing for both the main Microsoft Learn path and the retained Sentinel path:
1. **`configure_azure_monitor`** — sets up the OpenTelemetry pipeline that exports notebook traces to Application Insights
2. **`AIProjectInstrumentor().instrument()`** — enables preview GenAI tracing for project-backed agent creation and `responses.create(...)` calls
3. **`HTTPXClientInstrumentor().instrument()`** — enables HTTPX dependency instrumentation for the OpenAI/Responses client stack
4. **Explicit client spans around `responses.create(...)`** — emit the concrete notebook-side `HTTP` dependency rows that Service Map can correlate
5. **`OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental`** — opts into the latest experimental GenAI semantic conventions
6. **`AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION=true`** — makes trace-context propagation explicit for `get_openai_client()` traffic
7. **`AZURE_TRACING_GEN_AI_TRACE_CONTEXT_PROPAGATION_INCLUDE_BAGGAGE=true`** — propagates safe baggage keys used for run and agent correlation
8. **Explicit content-recording defaults** — keep prompt and tool payload capture off by default unless an operator opts in
9. **Manual notebook spans** — add run-specific context (`create_agent`, `invoke_agent`, `persist_story`, `persist_sentinel_query`) for correlation

**Note:**
- Both the main storytelling/Microsoft Learn flow and the Sentinel flow run through the Foundry Responses API with an `agent_reference` payload so they appear in Foundry Agent Traces.
- The notebook emits explicit notebook-side `POST /openai/v1/responses` dependencies so Azure Service Map has a direct client edge to correlate.
- Azure Monitor is initialized before Azure SDK tracing is enabled so notebook reruns are less likely to inherit a stale tracer provider in a reused kernel.
- Local notebook runs exclude managed identity from `DefaultAzureCredential` so the IMDS endpoint is not probed.

</details>

In [ ]:
import os
from html import escape
from uuid import uuid4

from azure.ai.projects.telemetry import AIProjectInstrumentor
from azure.core.settings import settings
from IPython.display import HTML, display
from opentelemetry import baggage, context as otel_context, trace
from opentelemetry.instrumentation.httpx import HTTPXClientInstrumentor

# -----------------------------------------------------------------------------
# Phase 1: Trace settings (must be set before instrumentation)
# -----------------------------------------------------------------------------
settings.tracing_implementation = None

project_name = foundry_proj_ep.rstrip("/").split("/")[-1] if "foundry_proj_ep" in globals() else "unknown-project"
os.environ["OTEL_SERVICE_NAME"] = "foundry-agent-framework-demo"
os.environ["OTEL_SERVICE_VERSION"] = "2026.04.06"
os.environ["OTEL_SEMCONV_STABILITY_OPT_IN"] = "gen_ai_latest_experimental"
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
os.environ["AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION"] = "true"
os.environ["AZURE_TRACING_GEN_AI_TRACE_CONTEXT_PROPAGATION_INCLUDE_BAGGAGE"] = "true"
os.environ["AZURE_TRACING_GEN_AI_INSTRUMENT_RESPONSES_API"] = "true"
os.environ.setdefault("OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT", "false")
os.environ.setdefault("AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED", "false")

if "telemetry_session_id" not in globals():
    telemetry_session_id = str(uuid4())

# -----------------------------------------------------------------------------
# Phase 2: Backend setup (Microsoft Foundry -> Application Insights connection)
# -----------------------------------------------------------------------------
if "running_on_azure_host" not in globals():
    azure_host_markers = (
        "WEBSITE_SITE_NAME",
        "WEBSITE_HOSTNAME",
        "FUNCTIONS_WORKER_RUNTIME",
        "KUBERNETES_SERVICE_HOST",
        "AKS_ARM_NAMESPACE_ID",
    )
    running_on_azure_host = any(os.environ.get(name) for name in azure_host_markers)

if not running_on_azure_host:
    os.environ["OTEL_EXPERIMENTAL_RESOURCE_DETECTORS"] = "otel"
    os.environ["OTEL_RESOURCE_DETECTORS"] = "otel"
    os.environ["APPLICATIONINSIGHTS_STATSBEAT_DISABLED_ALL"] = "true"

os.environ["OTEL_RESOURCE_ATTRIBUTES"] = (
    f"service.namespace=foundry-agent-demo,service.instance.id={telemetry_session_id},"
    f"deployment.environment=demo,foundry.project.name={project_name}"
)

from azure.monitor.opentelemetry import configure_azure_monitor

application_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

# -----------------------------------------------------------------------------
# Phase 3: Configure Azure Monitor + Foundry preview instrumentation
# -----------------------------------------------------------------------------
# Note: configure_azure_monitor sets up the Azure Monitor exporter path first so a
# reused notebook kernel does not leave the Azure SDK pointing at a stale provider.
if not globals().get("_project_otel_initialized", False):
    configure_azure_monitor(
        connection_string=application_insights_connection_string,
        resource=create_resource(),
        sampling_ratio=1.0,
        disable_logging=True,
        disable_metrics=True,
        enable_performance_counters=False,
    )

    settings.tracing_implementation = "opentelemetry"

    try:
        AIProjectInstrumentor().uninstrument()
    except Exception:
        pass

    AIProjectInstrumentor().instrument()
    globals()["_project_otel_initialized"] = True
else:
    settings.tracing_implementation = "opentelemetry"
    print("ℹ️ Foundry client-side telemetry was already initialized in this kernel session.")

if not globals().get("_httpx_otel_initialized", False):
    try:
        HTTPXClientInstrumentor().uninstrument()
    except Exception:
        pass

    HTTPXClientInstrumentor().instrument()
    globals()["_httpx_otel_initialized"] = True
else:
    print("ℹ️ HTTPX dependency instrumentation was already initialized in this kernel session.")

# -----------------------------------------------------------------------------
# Phase 4: Tracer and baggage context setup
# -----------------------------------------------------------------------------
tracer = trace.get_tracer("foundry_agent_framework_notebook")


def make_baggage_context(values: dict[str, object]):
    context = otel_context.get_current()
    for key, value in values.items():
        if value is None:
            continue
        value_text = str(value).strip()
        if value_text:
            context = baggage.set_baggage(key, value_text, context=context)
    return context


# -----------------------------------------------------------------------------
# Phase 5: Notebook status output
# -----------------------------------------------------------------------------
accent_style = "color: #C239B3; font-weight: 700;"
secondary_style = "color: #0078D4; font-weight: 700;"
local_mode_text = (
    "Local notebook mode - managed identity excluded; Azure resource detectors disabled"
    if not running_on_azure_host
    else "Azure-host environment detected - managed identity and Azure resource detection enabled"
)
content_recording_enabled = any(
    os.environ.get(name, "").strip().lower() in {"1", "true"}
    for name in (
        "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT",
        "AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED",
    )
)

display(
    HTML(
        f"""
<div style="font-family: Consolas, 'Cascadia Code', monospace; line-height: 1.5;">
  <div>Tracing enabled -&gt; Application Insights for project: '<span style=\"{accent_style}\">{escape(project_name)}</span>'</div>
  <div>- [connection string retrieved from: <span style=\"{accent_style}\">Microsoft Foundry</span>]</div>
  <div>- OTEL service name: <span style=\"{secondary_style}\">{escape(os.environ['OTEL_SERVICE_NAME'])}</span></div>
  <div>- Telemetry session id: {escape(str(telemetry_session_id))}</div>
  <div>- Foundry client tracing: Enabled via <span style=\"{secondary_style}\">AIProjectInstrumentor</span></div>
  <div>- HTTP client dependency tracing: Enabled via <span style=\"{secondary_style}\">HTTPXClientInstrumentor</span></div>
  <div>- GenAI semantic conventions: <span style=\"{secondary_style}\">latest experimental</span></div>
  <div>- Trace context propagation: Explicitly enabled</div>
  <div>- Baggage propagation: Enabled for notebook correlation keys</div>
  <div>- Message content recording: {"Enabled" if content_recording_enabled else "Disabled by default"}</div>
  <div>- Trace sampling: 100%</div>
  <div>- Runtime mode: {escape(local_mode_text)}</div>
</div>
"""
    )
)

<h3 style="color: #0078D4;">3.2 Configure MSFT Learn MCP Tool</h3>

Run this step to create the public remote MCP tool descriptor that the main Foundry project agent will use in Step 4.

<details>
<summary><strong>MSFT Learn MCP Tool Setup</strong></summary>

- MSFT Learn MCP documentation: [Azure MCP Server docs](https://learn.microsoft.com/azure/developer/azure-mcp-server/)
- MSFT Learn MCP URL: [Microsoft Learn MCP Endpoint](https://learn.microsoft.com/api/mcp)
- This step builds the public MCP tool payload with `MCPTool` from Azure AI Projects.
- The public Learn endpoint does not need a Foundry project connection, so the notebook can attach it directly to the main project agent.
- The Microsoft Sentinel MCP dependency remains separate in Step 3.3 because that path requires an existing Foundry project connection for OAuth passthrough.

</details>

In [ ]:
# ---------------------------------------------------------------------------
# MSFT Learn MCP Tool Spec (used by the Step 4 main project agent)
# ---------------------------------------------------------------------------
from azure.ai.projects.models import MCPTool

msft_learn_mcp_url = "https://learn.microsoft.com/api/mcp"
print(f"MSFT Learn MCP URL: {msft_learn_mcp_url}")

mcp_tool_spec = MCPTool(
    server_label="msft-learn",
    server_url=msft_learn_mcp_url,
)

print("Configured Foundry MCP tool: msft-learn")

<h3 style="color: #0078D4;">3.3 Configure Microsoft Sentinel Data Exploration MCP Tool</h3>

Run this step if you want the agent to use Microsoft Sentinel data exploration in addition to Microsoft Learn.

<details>
<summary><strong>Why this section is different</strong></summary>

- Microsoft Learn works as a public MCP endpoint, so the notebook can declare it directly.
- Microsoft Sentinel Data Exploration uses OAuth identity passthrough, so Foundry needs a project connection for the MCP endpoint.
- This cell uses `az rest` to look for an existing Foundry project connection that already points at the Sentinel MCP endpoint.
- If the connection exists, the notebook builds `sentinel_mcp_tool_spec` for Step 4.
- If it does not exist yet, the cell prints the exact project resource path and the expected connection name/ID so you can provision it and rerun the cell.

</details>

In [ ]:
# ---------------------------------------------------------------------------
# Microsoft Sentinel Data Exploration MCP Tool Spec (optional, OAuth passthrough)
# ---------------------------------------------------------------------------
import json
import os
import subprocess

from azure.ai.projects.models import MCPTool

az_cmd = "az.cmd" if os.name == "nt" else "az"
sentinel_mcp_url = "https://sentinel.microsoft.com/mcp/data-exploration"
sentinel_connection_name = os.environ.get("SENTINEL_MCP_CONNECTION_NAME", "MicrosoftSentinelData").strip()
sentinel_project_connection_id = os.environ.get("SENTINEL_MCP_PROJECT_CONNECTION_ID", "").strip()
sentinel_workspace_name = os.environ.get("SENTINEL_WORKSPACE_NAME", "DIBSecCom").strip()
sentinel_workspace_resource_group = os.environ.get("SENTINEL_WORKSPACE_RESOURCE_GROUP", "Sentinel").strip()
sentinel_subscription_name = os.environ.get("SENTINEL_SUBSCRIPTION_NAME", "Security").strip()
sentinel_subscription_id = ""
sentinel_workspace_resource_id = ""
sentinel_workspace_customer_id = ""
sentinel_mcp_tool_spec = None

def _run_az_json(command: list[str]) -> dict:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        stderr = (result.stderr or "").strip()
        raise RuntimeError(stderr or f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    stdout = (result.stdout or "").strip()
    return json.loads(stdout) if stdout else {}

def _run_az_tsv(command: list[str]) -> str:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        stderr = (result.stderr or "").strip()
        raise RuntimeError(stderr or f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    return (result.stdout or "").strip()

try:
    sentinel_subscription_id = _run_az_tsv(
        [
            az_cmd,
            "account",
            "list",
            "--query",
            f"[?name=='{sentinel_subscription_name}'].id | [0]",
            "-o",
            "tsv",
        ]
    )
except Exception:
    sentinel_subscription_id = ""

if sentinel_subscription_id:
    try:
        sentinel_workspace_resource_id = _run_az_tsv(
            [
                az_cmd,
                "monitor",
                "log-analytics",
                "workspace",
                "show",
                "--resource-group",
                sentinel_workspace_resource_group,
                "--workspace-name",
                sentinel_workspace_name,
                "--subscription",
                sentinel_subscription_id,
                "--query",
                "id",
                "-o",
                "tsv",
            ]
        )
        sentinel_workspace_customer_id = _run_az_tsv(
            [
                az_cmd,
                "monitor",
                "log-analytics",
                "workspace",
                "show",
                "--resource-group",
                sentinel_workspace_resource_group,
                "--workspace-name",
                sentinel_workspace_name,
                "--subscription",
                sentinel_subscription_id,
                "--query",
                "customerId",
                "-o",
                "tsv",
            ]
        )
    except Exception as ex:
        print(f"⚠️ Unable to resolve Sentinel workspace identifiers: {type(ex).__name__}: {ex}")

foundry_account_id = _run_az_tsv(
    [
        az_cmd,
        "resource",
        "show",
        "--resource-group",
        build_info["rg"],
        "--name",
        build_info["foundry_name"],
        "--resource-type",
        "Microsoft.CognitiveServices/accounts",
        "--query",
        "id",
        "-o",
        "tsv",
    ]
)
project_resource_id = f"{foundry_account_id}/projects/{build_info['foundry_project_name']}"
connections_uri = f"https://management.azure.com{project_resource_id}/connections?api-version=2025-06-01"

print(f"Sentinel MCP URL: {sentinel_mcp_url}")
print(f"Foundry project resource: {project_resource_id}")
print(f"Sentinel workspace target: {sentinel_workspace_name} (subscription: {sentinel_subscription_name})")
if sentinel_subscription_id:
    print(f"Sentinel subscription ID: {sentinel_subscription_id}")
if sentinel_workspace_resource_id:
    print(f"Sentinel workspace resource ID: {sentinel_workspace_resource_id}")
if sentinel_workspace_customer_id:
    print(f"Sentinel workspace customer ID: {sentinel_workspace_customer_id}")

try:
    connections_response = _run_az_json([az_cmd, "rest", "--method", "get", "--uri", connections_uri])
    connections = connections_response.get("value", [])
except Exception as ex:
    connections = []
    print(f"⚠️ Unable to enumerate project connections via Azure CLI: {type(ex).__name__}: {ex}")

matching_connection = None
if sentinel_project_connection_id:
    matching_connection = next(
        (connection for connection in connections if connection.get("id", "").lower() == sentinel_project_connection_id.lower()),
        None,
    )

if matching_connection is None:
    matching_connection = next(
        (
            connection
            for connection in connections
            if connection.get("name", "").lower() == sentinel_connection_name.lower()
            or connection.get("properties", {}).get("target", "").rstrip("/").lower() == sentinel_mcp_url.rstrip("/").lower()
        ),
        None,
    )

if matching_connection is not None:
    sentinel_project_connection_id = matching_connection["id"]

if sentinel_project_connection_id:
    sentinel_mcp_tool_spec = MCPTool(
        server_label="microsoft-sentinel-data",
        server_url=sentinel_mcp_url,
        require_approval="always",
        project_connection_id=sentinel_project_connection_id,
    )
    globals()["sentinel_workspace_name"] = sentinel_workspace_name
    globals()["sentinel_workspace_resource_group"] = sentinel_workspace_resource_group
    globals()["sentinel_subscription_name"] = sentinel_subscription_name
    globals()["sentinel_subscription_id"] = sentinel_subscription_id
    globals()["sentinel_workspace_resource_id"] = sentinel_workspace_resource_id
    globals()["sentinel_workspace_customer_id"] = sentinel_workspace_customer_id
    print("✅ Microsoft Sentinel Data Exploration MCP is enabled for this notebook.")
    print(f"   Project connection ID: {sentinel_project_connection_id}")
else:
    print("⚠️ No Foundry project connection was found for the Sentinel MCP endpoint.")
    print(f"   Expected connection name: {sentinel_connection_name}")
    print(f"   CLI check: az rest --method get --uri \"{connections_uri}\"")
    print("   Set SENTINEL_MCP_PROJECT_CONNECTION_ID to an existing connection ID and rerun this cell.")
    print("   This notebook can attach the MCP tool once the Foundry project connection already exists.")

<h2 style="color: #0078D4;">4. Create the AI Agents</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">✨</td>
    <td style="border: none; padding: 4px 0; color: #555;">Main agent uses a Foundry project-backed Responses API path so storytelling and Microsoft Learn runs surface in Foundry Agent Traces</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔧</td>
    <td style="border: none; padding: 4px 0; color: #555;">MSFT Learn MCP is attached directly to the main project agent as a public MCP tool</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🛡️</td>
    <td style="border: none; padding: 4px 0; color: #555;">If a Foundry project connection exists, the notebook also creates a separate Sentinel project agent so the Sentinel MCP dependency stays on its existing project-connected path</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔄</td>
    <td style="border: none; padding: 4px 0; color: #555;">Both agent paths use <code style="color: #0078D4;">create_version</code> so the notebook can run repeatedly without recreating agents unnecessarily</td>
  </tr>
</table>

<p style="margin-top: 12px; color: #444;">
  <strong>Main agent</strong> = the general-purpose notebook agent for storytelling and Microsoft Learn questions.<br>
  <strong>Sentinel project agent</strong> = the optional specialist agent for Microsoft Sentinel data exploration through the retained project-connected MCP path.
</p>


In [ ]:
from uuid import uuid4
from opentelemetry.trace import SpanKind, Status, StatusCode

main_agent_name = "ZoDEfendersAgent-1702"
model_name = genai_model

# Main project agent: the default notebook agent for storytelling and grounded Microsoft Learn answers.
main_agent_instructions = (
    "You are an illuminating cybernetic storytelling agent. "
    "You craft engaging 6-sentence stories based on user prompts and context. "
    "Be sure to use vivid language and creative scenarios to captivate the reader. "
    "Use the full names of any characters you create in the story. "
    "You are also a helpful and practical assistant for learning Microsoft technologies. "
    "When the user asks factual Microsoft questions, use the tools available to you and ground the answer in Microsoft Learn."
 )

sentinel_workspace_name = globals().get("sentinel_workspace_name", "DIBSecCom")
sentinel_subscription_name = globals().get("sentinel_subscription_name", "Security")

# Sentinel project agent: an optional specialist for Microsoft Sentinel data exploration only.
sentinel_agent_instructions = (
    "You are a Expert Microsoft Sentinel Assistant. "
    "Use the connected Microsoft Sentinel MCP tool in natural language. "
    "Before any data exploration call, first use the list_sentinel_workspaces tool to resolve the available workspace IDs. "
    f"If the workspace '{sentinel_workspace_name}' in the '{sentinel_subscription_name}' subscription is available, select it and use its returned workspaceId in all subsequent Sentinel MCP calls. "
    "Use the signed-in user identity provided by the conversation context when looking up login activity. "
    "Do not invent workspace IDs nor write raw KQL unless the MCP tool requires it after schema discovery. "
    "Answer only with the fields the user asked for and keep the result short and factual."
 )

demo_run_id = globals().get("demo_run_id") or str(uuid4())
globals()["demo_run_id"] = demo_run_id

main_tool_labels = [getattr(mcp_tool_spec, "server_label", "unknown-tool")]
main_agent_creation_context = make_baggage_context(
    {
        "demo-run-id": demo_run_id,
        "agent-name": main_agent_name,
        "model-name": model_name,
        "project-name": globals().get("project_name", "unknown-project"),
        "telemetry-session-id": globals().get("telemetry_session_id", ""),
        "agent-runtime": "azure-ai-project-agent",
        "mcp-server": ",".join(main_tool_labels),
    }
)

# Create the main project agent with the Microsoft Learn MCP tool, and capture telemetry about the creation process and agent configuration.
with tracer.start_as_current_span(
    f"create_agent {main_agent_name}",
    kind=SpanKind.CLIENT,
    context=main_agent_creation_context,
 ) as span:
    span.set_attribute("gen_ai.operation.name", "create_agent")
    span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
    span.set_attribute("gen_ai.request.model", model_name)
    span.set_attribute("gen_ai.agent.name", main_agent_name)
    span.set_attribute("gen_ai.agent.description", main_agent_instructions)
    span.set_attribute("demo.run_id", demo_run_id)
    span.set_attribute("app.session.id", globals().get("telemetry_session_id", ""))
    span.set_attribute("agent.runtime", "azure.ai.projects")
    span.set_attribute("agent.tools.count", len(main_tool_labels))
    span.set_attribute("agent.tools.labels", ",".join(main_tool_labels))
    span.add_event("create_agent.start")

    try:
        # Note: if the agent definition is identical to a previous version, Foundry may return the existing agent version instead of creating a new one. 
        # This is expected behavior and will not affect the functionality of the agent.
        main_agent = project_client.agents.create_version(
            agent_name=main_agent_name,
            definition=PromptAgentDefinition(
                model=model_name,
                instructions=main_agent_instructions,
                tools=[mcp_tool_spec],
            ),
        )
        span.set_attribute("gen_ai.agent.id", main_agent.id)
        span.set_attribute("gen_ai.agent.version", str(main_agent.version))
        span.add_event("create_agent.success")
    except Exception as ex:
        span.record_exception(ex)
        span.set_attribute("error.type", type(ex).__name__)
        span.set_status(Status(StatusCode.ERROR, str(ex)))
        span.add_event("create_agent.failure")
        raise

main_agent_reference_payload = {
    "agent_reference": {
        "name": main_agent.name,
        "id": main_agent.id,
        "type": "agent_reference",
    }
}

globals()["main_agent"] = main_agent
globals()["main_agent_reference_payload"] = main_agent_reference_payload
globals()["main_tool_labels"] = main_tool_labels
print(
    f"Main project agent configured: {main_agent.name} "
    f"(id: {main_agent.id}, version: {main_agent.version})"
)
print(f"Attached tools: {', '.join(main_tool_labels)}")
print(f"Run correlation id: {demo_run_id}")

sentinel_project_agent = None
sentinel_agent_reference_payload = None
sentinel_tool = globals().get("sentinel_mcp_tool_spec")

# Create the Sentinel specialist only when the notebook finds a retained project connection for the Sentinel MCP endpoint.
if sentinel_tool is not None:
    sentinel_agent_name = f"{main_agent_name}-sentinel"
    sentinel_tool_labels = [getattr(sentinel_tool, "server_label", "microsoft-sentinel-data")]
    sentinel_creation_context = make_baggage_context(
        {
            "demo-run-id": demo_run_id,
            "agent-name": sentinel_agent_name,
            "model-name": model_name,
            "project-name": globals().get("project_name", "unknown-project"),
            "telemetry-session-id": globals().get("telemetry_session_id", ""),
            "agent-runtime": "azure-ai-project-agent",
            "mcp-server": ",".join(sentinel_tool_labels),
        }
    )

    with tracer.start_as_current_span(
        f"create_agent {sentinel_agent_name}",
        kind=SpanKind.CLIENT,
        context=sentinel_creation_context,
    ) as span:
        span.set_attribute("gen_ai.operation.name", "create_agent")
        span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
        span.set_attribute("gen_ai.request.model", model_name)
        span.set_attribute("gen_ai.agent.name", sentinel_agent_name)
        span.set_attribute("gen_ai.agent.description", sentinel_agent_instructions)
        span.set_attribute("demo.run_id", demo_run_id)
        span.set_attribute("app.session.id", globals().get("telemetry_session_id", ""))
        span.set_attribute("agent.runtime", "azure.ai.projects")
        span.set_attribute("agent.tools.count", 1)
        span.set_attribute("agent.tools.labels", ",".join(sentinel_tool_labels))
        span.add_event("create_agent.start")

        try:
            # Similar to the main agent, if an identical agent definition already exists, Foundry may return the existing version instead of creating a new one. 
            # This is expected and does not affect functionality.
            sentinel_project_agent = project_client.agents.create_version(
                agent_name=sentinel_agent_name,
                definition=PromptAgentDefinition(
                    model=model_name,
                    instructions=sentinel_agent_instructions,
                    tools=[sentinel_tool],
                ),
            )
            span.set_attribute("gen_ai.agent.id", sentinel_project_agent.id)
            span.set_attribute("gen_ai.agent.version", str(sentinel_project_agent.version))
            span.add_event("create_agent.success")
        except Exception as ex:
            span.record_exception(ex)
            span.set_attribute("error.type", type(ex).__name__)
            span.set_status(Status(StatusCode.ERROR, str(ex)))
            span.add_event("create_agent.failure")
            raise

    sentinel_agent_reference_payload = {
        "agent_reference": {
            "name": sentinel_project_agent.name,
            "id": sentinel_project_agent.id,
            "type": "agent_reference",
        }
    }

    globals()["sentinel_project_agent"] = sentinel_project_agent
    globals()["sentinel_agent_reference_payload"] = sentinel_agent_reference_payload
    globals()["sentinel_tool_labels"] = sentinel_tool_labels
    print(
        f"Sentinel project agent configured: {sentinel_project_agent.name} "
        f"(id: {sentinel_project_agent.id}, version: {sentinel_project_agent.version})"
    )
else:
    globals()["sentinel_project_agent"] = None
    globals()["sentinel_agent_reference_payload"] = None
    print("Sentinel project agent not created because no Foundry project connection is configured.")

<h2 style="color: #0078D4;">5. Query the Main Project Agent</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📖</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Pass 1</strong> — generate a fictional story from the main project-backed storytelling persona</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔍</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Pass 2</strong> — retrieve factual guidance from Microsoft Learn through the public MCP tool attached to the main project agent</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🧭</td>
    <td style="border: none; padding: 4px 0; color: #555;">This cell uses the Foundry Responses API with an <code style="color: #0078D4;">agent_reference</code> payload so the main flow appears in Foundry Agent Traces</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">💾</td>
    <td style="border: none; padding: 4px 0; color: #555;">This cell appends the story and Microsoft Learn results to <code style="color: #2EA043;">stories.json</code> with project-agent run metadata and writes a dark-themed Marp deck markdown file for presentation-friendly output</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;">Each interaction is wrapped in <code style="color: #0E7C6B;">OpenTelemetry</code> spans for trace correlation</td>
  </tr>
</table>


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

from IPython.display import Markdown, display
from opentelemetry import context as otel_context
from opentelemetry.trace import SpanKind, Status, StatusCode

if globals().get("main_agent") is None:
    raise RuntimeError("Run Section 4 first so the main project agent is configured.")

story_prompt = (
    "Write about a Cloud Solutions Architect - Security by the name of 'Azure Zo' in Frankfurt, Germany. "
    "He is a man who moonlights as a cybernetic superhero called 'Die DEfender', protector of the digital realm. "
    "Include a plot twist in the last sentence. Always refer to the two characters by their full names."
)

facts_prompt = (
    "Use the MSFT Learn MCP tool to explain what Microsoft Foundry is for a Cloud Solutions Architect - Security. "
    "Return concise factual guidance only with this exact structure:\n"
    "MSFT Learn Insights:\n"
    "- 3 to 5 key points\n"
    "- Security and governance considerations\n"
    "- Practical next steps\n"
    "Sources:\n"
    "- Include at least one Microsoft Learn URL"
)

main_agent = globals()["main_agent"]
main_agent_reference_payload = globals().get("main_agent_reference_payload") or {
    "agent_reference": {
        "name": main_agent.name,
        "id": main_agent.id,
        "type": "agent_reference",
    }
}
main_agent_display_name = getattr(main_agent, "name", globals().get("main_agent_name", "ZoDEfendersAgent-1702"))
main_agent_id = str(getattr(main_agent, "id", "") or "")
main_agent_version = str(getattr(main_agent, "version", "") or "")
main_tool_labels = globals().get(
    "main_tool_labels",
    [getattr(globals().get("mcp_tool_spec"), "server_label", "msft-learn")],
)

MAX_APPROVAL_ROUNDS = 5
run_id = str(globals().get("demo_run_id", "") or "")
session_id = str(globals().get("telemetry_session_id", "") or "")
project_name_value = str(globals().get("project_name", "unknown-project") or "unknown-project")
conversation_ids = globals().setdefault("conversation_ids", {})
generated_at = datetime.now(timezone.utc)
generated_at_iso = generated_at.isoformat()
generated_at_display = generated_at.strftime("%Y-%m-%d %H:%M:%SZ")
generated_at_token = generated_at.strftime("%Y%m%d-%H%M%S")
run_suffix = (run_id[:8] if run_id else "manual").lower()
marp_output_dir = Path("marp")
marp_output_path = marp_output_dir / f"main-project-agent-{generated_at_token}-{run_suffix}.md"
stories_file = Path("stories.json")


def parse_response(response) -> tuple[str, list[str], list[str], object]:
    output_items = getattr(response, "output", None) or []
    output_types = [getattr(item, "type", type(item).__name__) for item in output_items]

    approval_ids = [
        item.id
        for item in output_items
        if getattr(item, "type", None) == "mcp_approval_request" and getattr(item, "id", None)
    ]

    text_parts = []
    for item in output_items:
        for content in getattr(item, "content", None) or []:
            text_obj = getattr(content, "text", None)
            if isinstance(text_obj, str) and text_obj.strip():
                text_parts.append(text_obj)
                continue

            text_value = getattr(text_obj, "value", None)
            if isinstance(text_value, str) and text_value.strip():
                text_parts.append(text_value)

    text = (getattr(response, "output_text", None) or "\n".join(text_parts)).strip()
    return text, approval_ids, output_types, getattr(response, "incomplete_details", None)


def build_responses_url(openai_client) -> str:
    response_base_url = str(getattr(openai_client, "base_url", foundry_proj_ep)).rstrip("/") + "/"
    if response_base_url.rstrip("/").endswith("/openai/v1") or "/openai/v1/" in response_base_url:
        return urljoin(response_base_url, "responses")
    return urljoin(response_base_url, "openai/v1/responses")


def run_query_with_auto_approval(openai_client, agent_reference_payload, prompt, interaction_name, interaction_span, max_rounds=5):
    conversation = openai_client.conversations.create()
    print(f"Conversation created: {conversation.id}")
    interaction_span.add_event(
        "conversation.created",
        {
            "app.conversation.id": conversation.id,
            "app.interaction": interaction_name,
        },
    )

    baggage_values = {
        "demo-run-id": run_id,
        "agent-name": main_agent_display_name,
        "agent-id": main_agent_id,
        "agent-version": main_agent_version,
        "interaction-name": interaction_name,
        "conversation-id": conversation.id,
        "model-name": model_name,
        "project-name": project_name_value,
        "telemetry-session-id": session_id,
        "agent-runtime": "azure-ai-project-agent",
        "mcp-server": ",".join(main_tool_labels),
    }

    def create_agent_response(response_input):
        baggage_token = otel_context.attach(make_baggage_context(baggage_values))
        responses_url = build_responses_url(openai_client)
        parsed_url = urlparse(responses_url)

        with tracer.start_as_current_span("POST /openai/v1/responses", kind=SpanKind.CLIENT) as response_http_span:
            response_http_span.set_attribute("http.request.method", "POST")
            response_http_span.set_attribute("url.full", responses_url)
            if parsed_url.hostname:
                response_http_span.set_attribute("server.address", parsed_url.hostname)
            if parsed_url.scheme:
                response_http_span.set_attribute("url.scheme", parsed_url.scheme)
            response_http_span.set_attribute("gen_ai.operation.name", "responses.create")
            response_http_span.set_attribute("gen_ai.agent.name", main_agent_display_name)

            try:
                response = openai_client.responses.create(
                    conversation=conversation.id,
                    input=response_input,
                    extra_body=agent_reference_payload,
                )
                response_http_span.set_attribute("http.response.status_code", 200)
                return response
            except Exception as ex:
                response_http_span.record_exception(ex)
                response_http_span.set_attribute("error.type", type(ex).__name__)
                response_http_span.set_status(Status(StatusCode.ERROR, str(ex)))
                raise
            finally:
                otel_context.detach(baggage_token)

    response = create_agent_response(prompt)
    approvals_granted = 0

    for round_number in range(1, max_rounds + 1):
        text, approval_ids, output_types, incomplete_details = parse_response(response)
        status = str(getattr(response, "status", "unknown") or "unknown")

        print(f"Response status: {status}")
        print(f"Response output types: {output_types}")
        if incomplete_details:
            print(f"Incomplete details: {incomplete_details}")
            interaction_span.add_event(
                "response.incomplete",
                {"app.incomplete_details": str(incomplete_details)},
            )

        interaction_span.set_attribute("app.response.status", status)
        interaction_span.set_attribute("app.response.output_types", ",".join(output_types))

        if text or not approval_ids:
            interaction_span.add_event(
                "response.completed",
                {
                    "gen_ai.response.id": str(getattr(response, "id", "") or ""),
                    "app.approval.rounds": approvals_granted,
                },
            )
            return conversation.id, response, text, approvals_granted

        approvals_granted += len(approval_ids)
        interaction_span.add_event(
            "mcp.approval.auto_approved",
            {
                "app.approval.round": round_number,
                "app.approval.count": len(approval_ids),
            },
        )
        print(f"Approving MCP tool requests (round {round_number}): {approval_ids}")
        response = create_agent_response(
            [
                {
                    "type": "mcp_approval_response",
                    "approve": True,
                    "approval_request_id": request_id,
                }
                for request_id in approval_ids
            ]
        )

    final_text, _, _, _ = parse_response(response)
    interaction_span.add_event(
        "response.max_approval_rounds_reached",
        {"app.approval.rounds": approvals_granted},
    )
    print("⚠️ Reached maximum MCP approval rounds without assistant text output.")
    return conversation.id, response, final_text, approvals_granted


def run_interaction_with_span(*, openai_client, interaction_name: str, prompt: str):
    interaction_context = make_baggage_context(
        {
            "demo-run-id": run_id,
            "agent-name": main_agent_display_name,
            "agent-id": main_agent_id,
            "agent-version": main_agent_version,
            "interaction-name": interaction_name,
            "model-name": model_name,
            "project-name": project_name_value,
            "telemetry-session-id": session_id,
            "agent-runtime": "azure-ai-project-agent",
            "mcp-server": ",".join(main_tool_labels),
        }
    )

    with tracer.start_as_current_span(
        f"invoke_agent {main_agent_display_name}",
        kind=SpanKind.CLIENT,
        context=interaction_context,
    ) as interaction_span:
        if run_id:
            interaction_span.set_attribute("demo.run_id", run_id)
        interaction_span.set_attribute("gen_ai.operation.name", "invoke_agent")
        interaction_span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
        interaction_span.set_attribute("gen_ai.request.model", model_name)
        interaction_span.set_attribute("gen_ai.agent.name", main_agent_display_name)
        interaction_span.set_attribute("gen_ai.agent.id", main_agent_id)
        interaction_span.set_attribute("gen_ai.agent.version", main_agent_version)
        interaction_span.set_attribute("app.interaction", interaction_name)
        interaction_span.set_attribute("app.prompt", prompt)
        interaction_span.set_attribute("app.session.id", session_id)
        interaction_span.set_attribute("agent.runtime", "azure.ai.projects")
        interaction_span.set_attribute("agent.tools.labels", ",".join(main_tool_labels))

        try:
            conversation_id, response, text, approvals_granted = run_query_with_auto_approval(
                openai_client=openai_client,
                agent_reference_payload=main_agent_reference_payload,
                prompt=prompt,
                interaction_name=interaction_name,
                interaction_span=interaction_span,
                max_rounds=MAX_APPROVAL_ROUNDS,
            )
            conversation_ids[interaction_name] = conversation_id
            interaction_span.set_attribute("app.conversation.id", conversation_id)
            interaction_span.set_attribute("app.approval.rounds", approvals_granted)
            interaction_span.set_attribute("app.completion", text[:500] if text else "")
            interaction_span.set_attribute("gen_ai.response.id", str(getattr(response, "id", "") or ""))
            response_model = getattr(response, "model", None)
            if response_model:
                interaction_span.set_attribute("gen_ai.response.model", response_model)
            return response, text, conversation_id
        except Exception as ex:
            interaction_span.record_exception(ex)
            interaction_span.set_attribute("error.type", type(ex).__name__)
            interaction_span.set_status(Status(StatusCode.ERROR, str(ex)))
            interaction_span.add_event("response.failure")
            raise


def append_story(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            stories = json.load(file)
        if not isinstance(stories, list):
            stories = []
    except (FileNotFoundError, json.JSONDecodeError):
        stories = []

    next_id = max((item.get("id", 0) for item in stories), default=0) + 1
    payload["id"] = next_id
    stories.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(stories, file, indent=2, ensure_ascii=False)

    return next_id


def strip_heading(text: str, heading: str) -> str:
    lines = (text or "").strip().splitlines()
    if lines and lines[0].strip().rstrip(":").casefold() == heading.rstrip(":").casefold():
        lines = lines[1:]
        while lines and not lines[0].strip():
            lines = lines[1:]
    return "\n".join(lines).strip()


def normalize_marp_text(text: str) -> str:
    cleaned_lines = []
    for raw_line in (text or "").splitlines():
        line = raw_line.rstrip()
        if line.strip() == "---":
            line = "\\---"
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines).strip()


def build_marp_deck(*, story_text: str, facts_text: str, conversation_map: dict[str, str], story_id: int, marp_path: Path, stories_path: Path) -> str:
    story_body = normalize_marp_text(story_text) or "_No story output returned._"
    facts_body = normalize_marp_text(strip_heading(facts_text, "MSFT Learn Insights")) or "_No Microsoft Learn output returned._"
    conversation_lines = [
        f"- {name.title()} conversation: `{conversation_id}`"
        for name, conversation_id in conversation_map.items()
    ]
    if not conversation_lines:
        conversation_lines = ["- Conversation ids unavailable"]

    deck_lines = [
        "---",
        "marp: true",
        "paginate: true",
        "theme: default",
        "style: |",
        "  section {",
        "    font-family: 'Aptos', 'Segoe UI', sans-serif;",
        "    background:",
        "      radial-gradient(circle at top right, rgba(56, 189, 248, 0.18), transparent 32%),",
        "      radial-gradient(circle at bottom left, rgba(96, 165, 250, 0.16), transparent 30%),",
        "      linear-gradient(145deg, #020617 0%, #0f172a 52%, #172554 100%);",
        "    color: #e2e8f0;",
        "    padding: 44px 54px;",
        "    font-size: 30px;",
        "    line-height: 1.38;",
        "  }",
        "  section.lead {",
        "    justify-content: center;",
        "    text-align: left;",
        "  }",
        "  section.story-slide {",
        "    padding: 34px 42px;",
        "    font-size: 21px;",
        "    line-height: 1.28;",
        "  }",
        "  section.learn-slide {",
        "    padding: 28px 34px;",
        "    font-size: 15px;",
        "    line-height: 1.14;",
        "  }",
        "  section.learn-slide h2 {",
        "    font-size: 1.25em;",
        "    margin-bottom: 0.2em;",
        "  }",
        "  section.learn-slide ul,",
        "  section.learn-slide ol,",
        "  section.learn-slide p {",
        "    font-size: 1em;",
        "    line-height: 1.16;",
        "  }",
        "  h1, h2, h3 {",
        "    color: #f8fafc;",
        "    letter-spacing: -0.02em;",
        "  }",
        "  h1 {",
        "    font-size: 2.0em;",
        "    margin-bottom: 0.2em;",
        "  }",
        "  h2 {",
        "    border-bottom: 1px solid rgba(148, 163, 184, 0.35);",
        "    padding-bottom: 0.18em;",
        "  }",
        "  strong {",
        "    color: #7dd3fc;",
        "  }",
        "  code {",
        "    background: rgba(15, 23, 42, 0.9);",
        "    color: #bfdbfe;",
        "    border: 1px solid rgba(125, 211, 252, 0.22);",
        "    border-radius: 8px;",
        "    padding: 0.12em 0.35em;",
        "  }",
        "  a {",
        "    color: #93c5fd;",
        "  }",
        "  ul, ol {",
        "    font-size: 0.82em;",
        "    line-height: 1.45;",
        "  }",
        "  p {",
        "    font-size: 0.9em;",
        "  }",
        "  section::after {",
        "    color: rgba(226, 232, 240, 0.55);",
        "    font-size: 0.55em;",
        "  }",
        "---",
        "<!-- _class: lead -->",
        "# Fictional Story + Microsoft Learn",
        "",
        "### Foundry project-backed Responses API path with Microsoft Learn MCP",
        "",
        f"**Story ID:** `{story_id}`  ",
        f"**Agent:** `{main_agent_display_name}`  ",
        f"**Generated:** `{generated_at_display}`",
        "",
        "---",
        "<!-- _class: story-slide -->",
        "## Fictional Story",
        "",
        story_body,
        "",
        "---",
        "<!-- _class: learn-slide -->",
        "## Microsoft Learn Insights",
        "",
        facts_body,
        "",
        "---",
        "## Run Metadata",
        "",
        *conversation_lines,
        f"- Record saved: `{story_id}` in `{stories_path.as_posix()}`",
        f"- Marp deck: `{marp_path.as_posix()}`",
        "- Runtime: `responses.create + agent_reference`",
        f"- Project agent: `{main_agent_display_name}`",
    ]
    return "\n".join(deck_lines) + "\n"


def render_query_marp_status(marp_path: Path, story_id: int, conversation_map: dict[str, str]) -> None:
    conversation_lines = [
        f"- {name.title()} conversation: `{conversation_id}`"
        for name, conversation_id in conversation_map.items()
    ]
    if not conversation_lines:
        conversation_lines = ["- Conversation ids unavailable"]

    marp_uri = marp_path.resolve().as_uri()
    status_markdown = "\n".join(
        [
            "### Marp deck created",
            "",
            f"- Deck: [{marp_path.as_posix()}]({marp_uri})",
            f"- Story id: `{story_id}`",
            "- Runtime: `responses.create + agent_reference`",
            *conversation_lines,
            "",
            "_Open the generated markdown file in VS Code and use Marp: Open Preview to view the dark-themed deck._",
        ]
    )
    display(Markdown(status_markdown))


with project_client.get_openai_client() as openai_client:
    print("\n=== Pass 1: Generate fictional story (project-backed main agent) ===")
    story_response, story_text, story_conversation_id = run_interaction_with_span(
        openai_client=openai_client,
        interaction_name="story",
        prompt=story_prompt,
    )

    print("\n=== Pass 2: Retrieve factual Microsoft Learn insights (project-backed main agent + MCP) ===")
    facts_response, facts_text, facts_conversation_id = run_interaction_with_span(
        openai_client=openai_client,
        interaction_name="facts",
        prompt=facts_prompt,
    )

output_sections = []
if story_text:
    output_sections.append("Fictional Story:\n" + story_text.strip())
if facts_text:
    output_sections.append(facts_text.strip())

assistant_text = "\n\n".join(section for section in output_sections if section).strip()
conversation_id_map = {
    "story": story_conversation_id,
    "facts": facts_conversation_id,
}

persist_context = make_baggage_context(
    {
        "demo-run-id": run_id,
        "agent-name": main_agent_display_name,
        "agent-id": main_agent_id,
        "agent-version": main_agent_version,
        "model-name": model_name,
        "project-name": project_name_value,
        "telemetry-session-id": session_id,
        "agent-runtime": "azure-ai-project-agent",
        "mcp-server": ",".join(main_tool_labels),
    }
)

with tracer.start_as_current_span("persist_story", context=persist_context) as persist_span:
    story_record = {
        "entry_type": "main_project_agent",
        "timestamp": generated_at_iso,
        "agent": main_agent_display_name,
        "agent_id": main_agent_id,
        "agent_version": main_agent_version,
        "agent_runtime": "azure-ai-project-agent",
        "model": model_name,
        "prompt": story_prompt,
        "story": story_text,
        "msft_learn_insights": facts_text,
        "combined_output": assistant_text,
        "tool_labels": main_tool_labels,
        "conversation_ids": conversation_id_map,
        "foundry_project_endpoint": build_info["foundry_project_endpoint"],
        "resource_group": build_info["rg"],
        "requested_by": build_info.get("requested_by"),
        "demo_run_id": run_id,
        "marp_output_file": marp_output_path.as_posix(),
    }
    story_id = append_story(stories_file, story_record)
    persist_span.set_attribute("app.story.id", story_id)
    persist_span.set_attribute("app.stories.path", str(stories_file))
    persist_span.set_attribute("app.interaction.count", len(conversation_id_map))
    persist_span.set_attribute("app.marp.path", marp_output_path.as_posix())

marp_output_dir.mkdir(parents=True, exist_ok=True)
marp_deck = build_marp_deck(
    story_text=story_text,
    facts_text=facts_text,
    conversation_map=conversation_id_map,
    story_id=story_id,
    marp_path=marp_output_path,
    stories_path=stories_file,
)
marp_output_path.write_text(marp_deck, encoding="utf-8")
globals()["latest_marp_output_path"] = marp_output_path
print(f"Marp deck written to {marp_output_path.resolve()}")

if not assistant_text:
    print("⚠️ No assistant text was returned.")
else:
    render_query_marp_status(marp_output_path, story_id, conversation_id_map)

<h3 style="color: #0078D4;">5.1 Query Microsoft Sentinel Data Exploration MCP</h3>

This call remains separate from the main storytelling and Microsoft Learn path because the Sentinel flow intentionally stays on the Foundry project-connected MCP dependency.

<details>
<summary><strong>What this cell does</strong></summary>

- Uses the Sentinel-specific project agent created in Step 4 when a Foundry project connection is available.
- Keeps the Microsoft Sentinel MCP dependency on the existing project connection and OAuth passthrough path by design.
- Wraps the interaction in dedicated OpenTelemetry spans so the Sentinel trace is easy to isolate in Foundry and Log Analytics.
- Persists the result to <code style="color: #2EA043;">stories.json</code> as a separate notebook record.
- Writes a single-slide Marp deck markdown file with a rustic-orange-to-dark background for presentation-friendly Sentinel output.
- Handles MCP approvals and OAuth consent inside this cell so it does not depend on the main project-agent query helpers.

</details>

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

from IPython.display import Markdown, display
from opentelemetry import context as otel_context
from opentelemetry.trace import SpanKind, Status, StatusCode

if globals().get("sentinel_project_agent") is None:
    raise RuntimeError("Run Section 4 first so the Sentinel project agent is configured.")

sentinel_target_upn = (
    os.environ.get("SENTINEL_UPN", "").strip()
    or str(globals().get("signed_in_account", "") or "").strip()
)
if not sentinel_target_upn:
    raise RuntimeError(
        "No signed-in user identity is available for the Sentinel query. "
        "Run Section 3 again or set SENTINEL_UPN before running this cell."
    )

sentinel_workspace_name = globals().get("sentinel_workspace_name", "DIBSecCom")
sentinel_subscription_name = globals().get("sentinel_subscription_name", "Security")
sentinel_workspace_id = str(
    globals().get("sentinel_workspace_customer_id", "")
    or globals().get("sentinel_workspace_resource_id", "")
    or ""
).strip()
sentinel_prompt = "".join(
    [
        "Use Microsoft Sentinel MCP in natural language. ",
        "First call list_sentinel_workspaces. ",
        f"Resolve the workspace ID for workspace '{sentinel_workspace_name}' in the '{sentinel_subscription_name}' subscription. ",
        (
            f"Use workspace ID parameter '{sentinel_workspace_id}' as a resolution hint when selecting the workspace. "
            if sentinel_workspace_id
            else ""
        ),
        "Use that returned workspaceId for all subsequent Sentinel MCP calls in this run. ",
        f"Use my signed-in identity '{sentinel_target_upn}'. ",
        "Tell me my most recent sign-in and where it happened. ",
        "Return only these fields: IP, Location, City, State, Country, Date, UPN, Logged Into. ",
        "If the workspace cannot be resolved or there is no data, say unavailable for each field.",
    ]
)

sentinel_project_agent = globals()["sentinel_project_agent"]
sentinel_agent_reference_payload = globals().get("sentinel_agent_reference_payload") or {
    "agent_reference": {
        "name": sentinel_project_agent.name,
        "id": sentinel_project_agent.id,
        "type": "agent_reference",
    }
}
sentinel_agent_display_name = getattr(
    sentinel_project_agent,
    "name",
    globals().get("sentinel_project_agent_name", "ZoDEfendersAgent-1702-sentinel"),
)
sentinel_agent_id = str(getattr(sentinel_project_agent, "id", "") or "")
sentinel_agent_version = str(getattr(sentinel_project_agent, "version", "") or "")
sentinel_tool_labels = globals().get("sentinel_tool_labels", ["microsoft-sentinel-data"])

run_id = str(globals().get("demo_run_id", "") or "")
session_id = str(globals().get("telemetry_session_id", "") or "")
project_name_value = str(globals().get("project_name", "unknown-project") or "unknown-project")
conversation_ids = globals().setdefault("conversation_ids", {})
generated_at = datetime.now(timezone.utc)
generated_at_iso = generated_at.isoformat()
generated_at_display = generated_at.strftime("%Y-%m-%d %H:%M:%SZ")
generated_at_token = generated_at.strftime("%Y%m%d-%H%M%S")
run_suffix = (run_id[:8] if run_id else "manual").lower()
marp_output_dir = Path("marp")
marp_output_path = marp_output_dir / f"sentinel-project-agent-{generated_at_token}-{run_suffix}.md"
stories_file = Path("stories.json")


def parse_response(response) -> tuple[str, list[str]]:
    output_items = getattr(response, "output", None) or []
    approval_ids = [
        item.id
        for item in output_items
        if getattr(item, "type", None) == "mcp_approval_request" and getattr(item, "id", None)
    ]

    text_parts = []
    for item in output_items:
        for content in getattr(item, "content", None) or []:
            text_obj = getattr(content, "text", None)
            if isinstance(text_obj, str) and text_obj.strip():
                text_parts.append(text_obj)
                continue
            text_value = getattr(text_obj, "value", None)
            if isinstance(text_value, str) and text_value.strip():
                text_parts.append(text_value)

    text = (getattr(response, "output_text", None) or "\n".join(text_parts)).strip()
    return text, approval_ids


def build_responses_url(openai_client) -> str:
    response_base_url = str(getattr(openai_client, "base_url", foundry_proj_ep)).rstrip("/") + "/"
    if response_base_url.rstrip("/").endswith("/openai/v1") or "/openai/v1/" in response_base_url:
        return urljoin(response_base_url, "responses")
    return urljoin(response_base_url, "openai/v1/responses")


def append_story(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            stories = json.load(file)
        if not isinstance(stories, list):
            stories = []
    except (FileNotFoundError, json.JSONDecodeError):
        stories = []

    next_id = max((item.get("id", 0) for item in stories), default=0) + 1
    payload["id"] = next_id
    stories.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(stories, file, indent=2, ensure_ascii=False)

    return next_id


def normalize_marp_text(text: str) -> str:
    cleaned_lines = []
    for raw_line in (text or "").splitlines():
        line = raw_line.rstrip()
        if line.strip() == "---":
            line = "\\---"
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines).strip()


def build_marp_deck(*, response_text: str, conversation_id: str, story_id: int, marp_path: Path, stories_path: Path) -> str:
    response_body = normalize_marp_text(response_text) or "_No Sentinel response text was returned._"
    deck_lines = [
        "---",
        "marp: true",
        "paginate: true",
        "theme: default",
        "style: |",
        "  section {",
        "    font-family: 'Aptos', 'Segoe UI', sans-serif;",
        "    background:",
        "      radial-gradient(circle at top right, rgba(201, 118, 65, 0.28), transparent 26%),",
        "      radial-gradient(circle at bottom left, rgba(120, 57, 31, 0.22), transparent 32%),",
        "      linear-gradient(145deg, #7a3e1d 0%, #3d2116 44%, #110c0a 100%);",
        "    color: #f8fafc;",
        "    padding: 44px 54px;",
        "    font-size: 30px;",
        "    line-height: 1.38;",
        "  }",
        "  section.lead {",
        "    justify-content: center;",
        "    text-align: left;",
        "  }",
        "  section.sentinel-slide {",
        "    padding: 30px 36px;",
        "  }",
        "  section.sentinel-slide h2 {",
        "    font-size: 1.25em;",
        "    margin-bottom: 0.2em;",
        "  }",
        "  section.sentinel-slide p,",
        "  section.sentinel-slide ul,",
        "  section.sentinel-slide ol {",
        "    font-size: 1em;",
        "    line-height: 1.18;",
        "  }",
        "  h1, h2, h3 {",
        "    color: #ffffff;",
        "    letter-spacing: -0.02em;",
        "  }",
        "  h1 {",
        "    font-size: 2.0em;",
        "    margin-bottom: 0.2em;",
        "  }",
        "  h2 {",
        "    border-bottom: 1px solid rgba(255, 255, 255, 0.18);",
        "    padding-bottom: 0.18em;",
        "  }",
        "  strong {",
        "    color: #ffedd5;",
        "  }",
        "  code {",
        "    background: rgba(24, 17, 13, 0.8);",
        "    color: #fff7ed;",
        "    border: 1px solid rgba(255, 237, 213, 0.18);",
        "    border-radius: 8px;",
        "    padding: 0.12em 0.35em;",
        "  }",
        "  a {",
        "    color: #fde68a;",
        "  }",
        "  ul, ol {",
        "    font-size: 0.82em;",
        "    line-height: 1.4;",
        "  }",
        "  p {",
        "    font-size: 0.9em;",
        "  }",
        "  section::after {",
        "    color: rgba(255, 255, 255, 0.5);",
        "    font-size: 0.55em;",
        "  }",
        "---",
        "<!-- _class: lead -->",
        "# Sentinel DataLake MCP Result",
        "",
        "### Foundry project-backed Responses API path with Sentinel Data Lake MCP",
        "",
        f"**Story ID:** `{story_id}`  ",
        f"**Agent:** `{sentinel_agent_display_name}`  ",
        f"**Generated:** `{generated_at_display}`",
        "",
        "---",
        "<!-- _class: sentinel-slide -->",
        "## Sentinel DataLake MCP Result",
        "",
        response_body,
        "",
        "---",
        "## AI Agent Run Metadata",
        "",
        f"- Conversation id: `{conversation_id}`",
        f"- Workspace target: `{sentinel_workspace_name}` in `{sentinel_subscription_name}`",
        f"- Target identity: `{sentinel_target_upn}`",
        f"- Record saved: `{story_id}` in `{stories_path.as_posix()}`",
        f"- Marp deck: `{marp_path.as_posix()}`",
        "- Runtime: `responses.create + agent_reference`",
        f"- Project agent: `{sentinel_agent_display_name}`",
    ]
    return "\n".join(deck_lines) + "\n"


def render_query_marp_status(marp_path: Path, story_id: int, conversation_id: str) -> None:
    marp_uri = marp_path.resolve().as_uri()
    status_markdown = "\n".join(
        [
            "### Marp deck created",
            "",
            f"- Deck: [{marp_path.as_posix()}]({marp_uri})",
            f"- Story id: `{story_id}`",
            f"- Conversation id: `{conversation_id}`",
            f"- Workspace target: `{sentinel_workspace_name}` in `{sentinel_subscription_name}`",
            f"- Target identity: `{sentinel_target_upn}`",
            "- Runtime: `responses.create + agent_reference`",
            "",
            "_Open the generated markdown file in VS Code and use Marp: Open Preview to view the dark-themed deck._",
        ]
    )
    display(Markdown(status_markdown))


context = make_baggage_context(
    {
        "demo-run-id": run_id,
        "agent-name": sentinel_agent_display_name,
        "agent-id": sentinel_agent_id,
        "agent-version": sentinel_agent_version,
        "model-name": model_name,
        "project-name": project_name_value,
        "telemetry-session-id": session_id,
        "agent-runtime": "azure-ai-project-agent",
        "mcp-server": ",".join(sentinel_tool_labels),
        "target-upn": sentinel_target_upn,
        "sentinel-workspace": sentinel_workspace_name,
        "sentinel-subscription": sentinel_subscription_name,
    }
)

with tracer.start_as_current_span("sentinel-agent-query", kind=SpanKind.CLIENT, context=context) as span:
    span.set_attribute("gen_ai.operation.name", "invoke_agent")
    span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
    span.set_attribute("gen_ai.request.model", model_name)
    span.set_attribute("gen_ai.agent.name", sentinel_agent_display_name)
    span.set_attribute("gen_ai.agent.id", sentinel_agent_id)
    span.set_attribute("gen_ai.agent.version", sentinel_agent_version)
    span.set_attribute("app.session.id", session_id)
    span.set_attribute("app.prompt", sentinel_prompt)
    span.set_attribute("app.target_upn", sentinel_target_upn)
    span.set_attribute("app.sentinel.workspace", sentinel_workspace_name)
    span.set_attribute("app.sentinel.subscription", sentinel_subscription_name)

    try:
        with project_client.get_openai_client() as openai_client:
            conversation = openai_client.conversations.create()
            print(f"Conversation created: {conversation.id}")
            print(f"Sentinel workspace target: {sentinel_workspace_name} (preferred subscription: {sentinel_subscription_name})")
            print(f"Sentinel target identity: {sentinel_target_upn}")
            conversation_ids["sentinel"] = conversation.id

            def create_agent_response(response_input):
                baggage_token = otel_context.attach(context)
                responses_url = build_responses_url(openai_client)
                parsed_url = urlparse(responses_url)
                with tracer.start_as_current_span("POST /openai/v1/responses", kind=SpanKind.CLIENT) as response_http_span:
                    response_http_span.set_attribute("http.request.method", "POST")
                    response_http_span.set_attribute("url.full", responses_url)
                    if parsed_url.hostname:
                        response_http_span.set_attribute("server.address", parsed_url.hostname)
                    if parsed_url.scheme:
                        response_http_span.set_attribute("url.scheme", parsed_url.scheme)
                    response_http_span.set_attribute("gen_ai.operation.name", "responses.create")
                    response_http_span.set_attribute("gen_ai.agent.name", sentinel_agent_display_name)
                    try:
                        return openai_client.responses.create(
                            conversation=conversation.id,
                            input=response_input,
                            extra_body=sentinel_agent_reference_payload,
                        )
                    finally:
                        otel_context.detach(baggage_token)

            sentinel_response = create_agent_response(sentinel_prompt)
            sentinel_approval_rounds = 0
            sentinel_text = ""
            for round_number in range(1, 6):
                sentinel_text, approval_ids = parse_response(sentinel_response)
                if sentinel_text:
                    break
                if not approval_ids:
                    break
                sentinel_approval_rounds += len(approval_ids)
                print(f"Approving MCP tool requests (round {round_number}): {approval_ids}")
                sentinel_response = create_agent_response(
                    [
                        {
                            "type": "mcp_approval_response",
                            "approve": True,
                            "approval_request_id": approval_id,
                        }
                        for approval_id in approval_ids
                    ]
                )

        if not sentinel_text:
            sentinel_text = "No Sentinel response text was returned."

        story_record = {
            "entry_type": "sentinel_project_agent",
            "timestamp": generated_at_iso,
            "agent": sentinel_agent_display_name,
            "agent_id": sentinel_agent_id,
            "agent_version": sentinel_agent_version,
            "agent_runtime": "azure-ai-project-agent",
            "model": model_name,
            "prompt": sentinel_prompt,
            "story": sentinel_text,
            "target_upn": sentinel_target_upn,
            "workspace": sentinel_workspace_name,
            "subscription": sentinel_subscription_name,
            "tool_labels": sentinel_tool_labels,
            "conversation_ids": {"sentinel": conversation.id},
            "foundry_project_endpoint": build_info["foundry_project_endpoint"],
            "resource_group": build_info["rg"],
            "requested_by": build_info.get("requested_by"),
            "demo_run_id": run_id,
            "marp_output_file": marp_output_path.as_posix(),
        }
        story_id = append_story(stories_file, story_record)
        marp_output_dir.mkdir(parents=True, exist_ok=True)
        marp_deck = build_marp_deck(
            response_text=sentinel_text,
            conversation_id=conversation.id,
            story_id=story_id,
            marp_path=marp_output_path,
            stories_path=stories_file,
        )
        marp_output_path.write_text(marp_deck, encoding="utf-8")
        globals()["latest_sentinel_marp_output_path"] = marp_output_path
        print(f"Marp deck written to {marp_output_path.resolve()}")
        render_query_marp_status(marp_output_path, story_id, conversation.id)
    except Exception as ex:
        span.record_exception(ex)
        span.set_attribute("error.type", type(ex).__name__)
        span.set_status(Status(StatusCode.ERROR, str(ex)))
        raise

<h2 style="color: #0078D4;">6. Validate Observability (Traces) in Log Analytics</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">✅</td>
    <td style="border: none; padding: 4px 0; color: #555;">Confirm agentic telemetry is populating in Log Analytics</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔗</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>End-to-end view</strong> — joined dependencies and traces by <code style="color: #0078D4;">OperationId</code></td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📊</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Runs-only trend</strong> — call volume, failures, and latency percentiles in 15-min bins</td>
  </tr>
</table>

<details>
<summary><strong>Environment &amp; Query Prerequisites</strong></summary>
<br>

🗂️ <strong>Workspace resource ID</strong><br>
<code style="color: #888; font-size: 0.88em;">/subscriptions/&lt;subscription-id&gt;/resourceGroups/Sentinel/providers/Microsoft.OperationalInsights/workspaces/&lt;workspace-name&gt;</code><br><br>

⚠️ The Azure CLI command <code style="color: #D83B01;">az monitor log-analytics query</code> may fail in some environments due to extension/runtime mismatch.<br>
🔐 Fallback uses direct HTTPS calls to <code style="color: #0078D4;">api.loganalytics.io</code> and reuses the notebook's local credential policy so developer workstations do not probe the managed identity IMDS endpoint.

</details>


In [ ]:
import json
import subprocess
import os
import urllib.error
import urllib.request

from azure.identity import DefaultAzureCredential

# Dynamically resolve the Security subscription ID via Azure CLI
az_cmd = "az.cmd" if os.name == "nt" else "az"
workspace_subscription_id = subprocess.check_output(
    [az_cmd, "account", "list", "--query", "[?name=='Security'].id | [0]", "-o", "tsv"],
    text=True,
).strip()
workspace_resource_group = "Sentinel"
workspace_name = "DIBSecCom"
workspace_resource_id = (
    f"/subscriptions/{workspace_subscription_id}/resourceGroups/{workspace_resource_group}"
    f"/providers/Microsoft.OperationalInsights/workspaces/{workspace_name}"
)

# Dynamically resolve the workspace customer ID (required by Log Analytics Query API)
workspace_customer_id = subprocess.check_output(
    [az_cmd, "monitor", "log-analytics", "workspace", "show",
     "--resource-group", workspace_resource_group,
     "--workspace-name", workspace_name,
     "--subscription", workspace_subscription_id,
     "--query", "customerId", "-o", "tsv"],
    text=True,
).strip()
print(f"Using Log Analytics workspace: {workspace_resource_id}")
print(f"Workspace customer ID: {workspace_customer_id}")


def build_log_analytics_credential() -> DefaultAzureCredential:
    credential_factory = globals().get("_build_default_azure_credential")
    if callable(credential_factory):
        return credential_factory(prefer_local_credentials=True)

    local_running_on_azure_host = globals().get("running_on_azure_host")
    if local_running_on_azure_host is None:
        azure_host_markers = (
            "WEBSITE_SITE_NAME",
            "WEBSITE_HOSTNAME",
            "FUNCTIONS_WORKER_RUNTIME",
            "KUBERNETES_SERVICE_HOST",
            "AKS_ARM_NAMESPACE_ID",
        )
        local_running_on_azure_host = any(os.environ.get(name) for name in azure_host_markers)

    credential_kwargs = {}
    managed_identity_client_id = os.environ.get("AZURE_CLIENT_ID", "").strip()
    if local_running_on_azure_host and managed_identity_client_id:
        credential_kwargs["managed_identity_client_id"] = managed_identity_client_id

    if not local_running_on_azure_host:
        credential_kwargs["exclude_managed_identity_credential"] = True
        credential_kwargs["exclude_workload_identity_credential"] = True

    return DefaultAzureCredential(**credential_kwargs)


log_analytics_credential = build_log_analytics_credential()
if globals().get("running_on_azure_host"):
    print("Using Azure-host credential chain for Log Analytics query.")
else:
    print("Using local developer credential chain for Log Analytics query (managed identity excluded).")

# Optional: filter to one run when demo_run_id exists in the current kernel session
demo_run_filter = globals().get("demo_run_id")
if isinstance(demo_run_filter, str):
    demo_run_filter = demo_run_filter.strip()
else:
    demo_run_filter = ""
run_filter_clause = (
    f'| where tostring(Properties["demo.run_id"]) == "{demo_run_filter}"'
    if demo_run_filter
    else ""
)
if run_filter_clause:
    print(f"Applying run filter for demo.run_id={demo_run_filter}")
else:
    print("No demo_run_id filter applied (showing latest runs in lookback window).")

# End-to-end view including requested fields: agent name, model, system message, host, location
kql_end_to_end = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
__RUN_FILTER__
| where Name in ("agent-query", "responses") or Name startswith "create_agent " or Name startswith "invoke_agent " or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["gen_ai.agent.name"]),
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    system_message = coalesce(
        tostring(Properties["gen_ai.system_instructions"]),
        extract(@'"role"\s*:\s*"system".*?"content"\s*:\s*"([^\"]+)"', 1, tostring(Properties["gen_ai.input.messages"])),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    ),
    demo_run_id = tostring(Properties["demo.run_id"]),
    interaction = tostring(Properties["app.interaction"])
| project
    TimeGenerated,
    Name,
    demo_run_id,
    interaction,
    agent_name,
    model_used,
    system_message,
    computer,
    location,
    Success,
    DurationMs,
    OperationId,
    Prompt = coalesce(tostring(Properties["app.prompt"]), tostring(Properties["gen_ai.prompt"])),
    Completion = coalesce(tostring(Properties["app.completion"]), tostring(Properties["gen_ai.completion"]))
| order by TimeGenerated desc
| take 50
""".replace("__RUN_FILTER__", run_filter_clause).strip()

# Runs-only trend grouped by agent/model/host/location
kql_runs_trend = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
__RUN_FILTER__
| where Name in ("agent-query", "responses") or Name startswith "create_agent " or Name startswith "invoke_agent " or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["gen_ai.agent.name"]),
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    ),
    demo_run_id = tostring(Properties["demo.run_id"]),
    interaction = tostring(Properties["app.interaction"])
| summarize
    Calls = count(),
    Failures = countif(Success == false),
    AvgDurationMs = avg(DurationMs),
    P95DurationMs = percentile(DurationMs, 95)
    by bin(TimeGenerated, 15m), demo_run_id, interaction, agent_name, model_used, computer, location
| order by TimeGenerated desc
""".replace("__RUN_FILTER__", run_filter_clause).strip()

def query_log_analytics(customer_id: str, query: str) -> dict:
    token = log_analytics_credential.get_token("https://api.loganalytics.io/.default").token

    body = json.dumps({"query": query}).encode("utf-8")
    request = urllib.request.Request(
        url=f"https://api.loganalytics.io/v1/workspaces/{customer_id}/query",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token}",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as ex:
        error_body = ex.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Log Analytics query failed ({ex.code}): {error_body}") from ex

print("Running end-to-end validation query...")
end_to_end_result = query_log_analytics(workspace_customer_id, kql_end_to_end)
print(json.dumps(end_to_end_result, indent=2)[:4000])

print("\nRunning runs-only trend query...")
runs_trend_result = query_log_analytics(workspace_customer_id, kql_runs_trend)
print(json.dumps(runs_trend_result, indent=2)[:4000])